# Feature Store with Feast and Neo4j

This notebook builds a feature store using [Feast](https://feast.dev/) with Neo4j Desktop as the feature computation engine and SQLite as the online store. It also implements a graph-native feature (customer community membership) that has no SQL equivalent.

### Before you start

- Open Neo4j Desktop and ensure you have a project running.
- Create a new database called `feast`.
- The Graph Data Science (GDS) plugin needs to be installed.
    - Select the **three dots (...)** in the top-right.
    - Select **Plugins**.
    - Install **Graph Data Science Library**.

### Before running the notebook

Export the following in your shell:

```shell
export NEO4J_URI=bolt://127.0.0.1:7687
export NEO4J_USERNAME=neo4j
export NEO4J_PASSWORD=your_password_here
export NEO4J_DATABASE=feast
```

## 1. Install Dependencies

In [1]:
%pip install feast==0.66.0 \
             neo4j==6.2.0 \
             pandas==2.3.3 \
             pyarrow==25.0.1 \
             tabulate==0.10.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import pandas as pd
import random
import subprocess

from datetime import datetime, timedelta
from feast import FeatureStore
from neo4j import GraphDatabase
from tabulate import tabulate

## 3. Configuration

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

SEED = 42

print("Credentials set.")

Credentials set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity = "OFF"
)

print(driver.verify_connectivity()) # None is expected
print("Neo4j connection verified.")

None
Neo4j connection verified.


## 5. Create Feature Definitions

We write two files that Feast needs:

- `feature_store.yaml` -- project-level config, pointing at SQLite for the online store
- `features/customer_features.py` -- the entity and feature view declarations

The schema includes five standard e-commerce aggregations plus `community_id`, the graph-native feature we'll compute with GDS.

In [5]:
os.makedirs("feature_repo/data",     exist_ok = True)
os.makedirs("feature_repo/features", exist_ok = True)

feature_store_yaml = """
project: ecommerce_features
registry: feature_repo/registry.db
provider: local

online_store:
    type: sqlite
    path: feature_repo/online_store.db
"""

with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("Feature store configuration created.")

Feature store configuration created.


In [6]:
customer_features_py = """
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String, ValueType
from datetime import timedelta

customer = Entity(
    name = "customer_id",
    description = "Customer identifier",
    value_type = ValueType.STRING
)

customer_features = FeatureView(
    name = "customer_features",
    entities = [customer],
    ttl = timedelta(days = 90),
    schema = [
        Field(name = "total_orders",          dtype = Int64),
        Field(name = "total_spent",           dtype = Float32),
        Field(name = "avg_order_value",       dtype = Float32),
        Field(name = "days_since_last_order", dtype = Int64),
        Field(name = "favorite_category",     dtype = String),
        Field(name = "community_id",          dtype = Int64),
    ],
    online = True,
    source = FileSource(
        path = "data/customer_features.parquet",
        timestamp_field = "event_timestamp"
    ),
)
"""

with open("feature_repo/features/customer_features.py", "w") as f:
    f.write(customer_features_py)

print("Feature definitions created.")

Feature definitions created.


## 6. Load Sample Data into Neo4j

We generate 100 customers, 3,000 orders and 5 product categories. Each customer is assigned a preferred category, stored as a property on the `Customer` node. 80% of their orders come from that category and 20% are spread randomly across the others.

Storing the preferred category on the node lets us project a clean customer-to-customer similarity graph in GDS: two customers are connected if and only if they share the same preferred category. This gives Louvain a perfect signal with no noise and no threshold tuning required.

In [7]:
def setup_graph():
    """Clear any existing data and load fresh customer/order graph."""

    random.seed(SEED)

    customers  = [f"CUST_{i:04d}" for i in range(1, 101)]
    categories = ["Electronics", "Clothing", "Home", "Books", "Sports"]

    # Each customer has a preferred category -- 80% of their orders come from it
    preferred  = {cid: random.choice(categories) for cid in customers}

    orders_data = []
    for i in range(3000):
        cid      = random.choice(customers)
        category = preferred[cid] if random.random() < 0.8 else random.choice(categories)
        days_ago = random.randint(0, 90)
        orders_data.append({
            "customer_id":        cid,
            "preferred_category": preferred[cid],
            "order_id":           f"ORD_{i:06d}",
            "order_timestamp":    (datetime.now() - timedelta(
                                      days  = days_ago,
                                      hours = random.randint(0, 23)
                                   )).isoformat(),
            "order_value":        round(random.uniform(10, 500), 2),
            "product_category":   category,
            "quantity":           random.randint(1, 5),
        })

    with driver.session(database = NEO4J_DATABASE) as session:
        session.run("MATCH (n) DETACH DELETE n")

        session.run("""
            UNWIND $orders AS o
            MERGE (c:Customer {id: o.customer_id})
              ON CREATE SET c.preferred_category = o.preferred_category
            MERGE (cat:Category {name: o.product_category})
            CREATE (ord:Order {
                id:        o.order_id,
                value:     o.order_value,
                timestamp: datetime(o.order_timestamp),
                quantity:  o.quantity
            })
            MERGE (c)-[:PLACED]->(ord)
            MERGE (ord)-[:IN_CATEGORY]->(cat)
            MERGE (c)-[:PURCHASED_IN]->(cat)
        """, orders = orders_data)

    print(f"Loaded {len(orders_data)} orders for {len(customers)} customers.")

setup_graph()

Loaded 3000 orders for 100 customers.


## 7. Compute Standard Features with Cypher

There are five features. We compute them with two Cypher queries:

1. A main aggregation for counts, totals, averages and recency
2. A separate query for favorite category (requires finding the most frequent category per customer), which we then merge into the main DataFrame

In [8]:
def compute_standard_features():
    """Compute order-based aggregations for each customer."""

    with driver.session(database = NEO4J_DATABASE) as session:

        # Main aggregations
        result = session.run("""
            MATCH (c:Customer)-[:PLACED]->(o:Order)
            RETURN
                c.id                                               AS customer_id,
                count(o)                                           AS total_orders,
                sum(o.value)                                       AS total_spent,
                avg(o.value)                                       AS avg_order_value,
                duration.inDays(max(o.timestamp), datetime()).days AS days_since_last_order,
                max(o.timestamp)                                   AS event_timestamp
        """)
        df = pd.DataFrame([r.data() for r in result])
        df["event_timestamp"] = pd.to_datetime(
            df["event_timestamp"].apply(lambda t: t.iso_format() if t else None),
            utc = True
        )

        # Favorite category per customer
        result_cat = session.run("""
            MATCH (c:Customer)-[:PLACED]->(o:Order)-[:IN_CATEGORY]->(cat:Category)
            WITH c.id AS customer_id, cat.name AS category, count(*) AS cnt
            ORDER BY customer_id, cnt DESC
            WITH customer_id, collect(category)[0] AS favorite_category
            RETURN customer_id, favorite_category
        """)
        cat_df = pd.DataFrame([r.data() for r in result_cat])

    df = df.merge(cat_df, on = "customer_id")

    print(f"Computed standard features for {len(df)} customers.")
    print()
    sample = df.drop(columns = ["event_timestamp"]).head(4)
    print(tabulate(sample, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

    return df

features_df = compute_standard_features()

Computed standard features for 100 customers.

customer_id      total_orders    total_spent    avg_order_value    days_since_last_order  favorite_category
-------------  --------------  -------------  -----------------  -----------------------  -------------------
CUST_0032                  18        4237.33             235.41                        5  Home
CUST_0075                  32        8192.88             256.03                        1  Sports
CUST_0097                  30        7543.75             251.46                        7  Books
CUST_0077                  36        9464.27             262.90                        0  Clothing


## 8. Compute Graph-Native Feature: Community ID

We project a customer-to-customer similarity graph via Cypher, connecting customers who share the same preferred category. Because preferred category is stored as a node property, the projection is exact -- no threshold tuning, no noise. Louvain then detects one community per preferred category, producing five well-balanced groups.

In [9]:
def compute_community_feature():
    """Project a customer similarity graph by shared preferred category and run Louvain."""

    with driver.session(database = NEO4J_DATABASE) as session:

        # Drop the projection if it already exists
        session.run("CALL gds.graph.drop('customer-graph', false)")

        # Connect customers who share the same preferred category.
        # Using the stored node property gives an exact match with no noise.
        result = session.run("""
            CALL gds.graph.project.cypher(
                'customer-graph',
                'MATCH (c:Customer) RETURN id(c) AS id',
                'MATCH (c1:Customer), (c2:Customer)
                 WHERE id(c1) < id(c2)
                   AND c1.preferred_category = c2.preferred_category
                 RETURN id(c1) AS source, id(c2) AS target, 1 AS weight'
            )
            YIELD nodeCount, relationshipCount
            RETURN nodeCount, relationshipCount
        """)
        row = result.single()
        print(f"Projected graph: {row['nodeCount']} nodes, {row['relationshipCount']} relationships.")

        # Run Louvain on the customer similarity graph
        session.run("""
            CALL gds.louvain.write(
                'customer-graph',
                {writeProperty: 'community_id'}
            )
        """)

        # Drop the projection
        session.run("CALL gds.graph.drop('customer-graph', false)")

        # Fetch results ordered by community
        result = session.run("""
            MATCH (c:Customer)
            RETURN c.id                 AS customer_id,
                   c.community_id       AS community_id,
                   c.preferred_category AS preferred_category
            ORDER BY c.community_id, c.id
        """)
        community_df = pd.DataFrame([r.data() for r in result])

    print(f"Assigned community IDs to {len(community_df)} customers.")
    print(f"Number of communities: {community_df['community_id'].nunique()}")
    print()
    summary = (community_df
        .groupby(['community_id', 'preferred_category'])['customer_id']
        .count()
        .reset_index()
        .rename(columns = {'customer_id': 'customers'})
    )
    print(tabulate(summary, headers = "keys", tablefmt = "simple", showindex = False))

    return community_df[['customer_id', 'community_id']]

community_df = compute_community_feature()

Projected graph: 100 nodes, 985 relationships.
Assigned community IDs to 100 customers.
Number of communities: 5

  community_id  preferred_category      customers
--------------  --------------------  -----------
            91  Sports                         16
            92  Home                           22
            96  Clothing                       24
            98  Books                          15
            99  Electronics                    23


## 9. Merge Features and Write to Parquet

Feast's offline store is a Parquet file. We merge our standard and graph-native features into one DataFrame and write it out. From this point on, Feast handles the rest -- it doesn't know or care that some features came from Cypher and one came from GDS.

In [10]:
all_features_df = features_df.merge(community_df, on = "customer_id")

all_features_df.to_parquet("feature_repo/data/customer_features.parquet", index = False)

print(f"Wrote features for {len(all_features_df)} customers to Parquet.")
print()
sample = all_features_df.drop(columns = ["event_timestamp"]).head(4)
print(tabulate(sample, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

Wrote features for 100 customers to Parquet.

customer_id      total_orders    total_spent    avg_order_value    days_since_last_order  favorite_category      community_id
-------------  --------------  -------------  -----------------  -----------------------  -------------------  --------------
CUST_0032                  18        4237.33             235.41                        5  Home                             92
CUST_0075                  32        8192.88             256.03                        1  Sports                           91
CUST_0097                  30        7543.75             251.46                        7  Books                            98
CUST_0077                  36        9464.27             262.90                        0  Clothing                         96


## 10. Initialize Feast and Apply Feature Definitions

In [11]:
fs = FeatureStore(repo_path = "feature_repo")

print("Applying feature definitions...")

result = subprocess.run(
    ["feast", "-c", "feature_repo", "apply"],
    capture_output = True,
    text = True
)

print(result.stdout)

if result.returncode == 0:
    print("Features applied successfully.")
else:
    print("Error:", result.stderr)

Applying feature definitions...
No project found in the repository. Using project name ecommerce_features defined in feature_store.yaml
Applying changes for project ecommerce_features
Updated feature view customer_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
file_options {
  uri: "data/customer_features.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "data/customer_features.parquet"
meta {
  created_timestamp {
    seconds: 1788160888
    nanos: 636724000
  }
  last_updated_timestamp {
    seconds: 1788160888
    nanos: 645295000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
file_options {
  uri: "data/customer_features.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "data/customer_features.parquet"
meta {
  created_timestamp {
    seconds: 1788160929
    nanos: 532267000
  }
  last_updated_timestamp {
    seconds: 1788160929
    nanos: 532267000
  }


## 11. Materialize Features to the Online Store

Materialization reads the Parquet file and loads the features into the SQLite online store, where they can be retrieved at low latency during inference.

In [12]:
end_date   = datetime.now()
start_date = end_date - timedelta(days = 90)

print(f"Materializing features from {start_date} to {end_date}...")

result = subprocess.run(
    [
        "feast", "-c", "feature_repo", "materialize",
        start_date.isoformat(),
        end_date.isoformat()
    ],
    capture_output = True,
    text = True
)

print(result.stdout)

if result.returncode == 0:
    print("Features materialized to online store.")
else:
    print("Error:", result.stderr)

Materializing features from 2026-06-02 08:22:09.636035 to 2026-08-31 08:22:09.636035...
Materializing 1 feature views from 2026-06-02 08:22:09+00:00 to 2026-08-31 08:22:09+00:00 into the sqlite online store.

customer_features:

Features materialized to online store.


## 12. Online Feature Retrieval

We reinitialize the feature store and retrieve features for three test customers. Notice that `community_id` is retrieved alongside the standard features -- from Feast's perspective it's just another column.

In [13]:
fs = FeatureStore(repo_path = "feature_repo")

test_customers = ["CUST_0001", "CUST_0025", "CUST_0050"]
entity_rows    = [{"customer_id": cid} for cid in test_customers]

feature_vector = fs.get_online_features(
    features = [
        "customer_features:total_orders",
        "customer_features:total_spent",
        "customer_features:avg_order_value",
        "customer_features:days_since_last_order",
        "customer_features:favorite_category",
        "customer_features:community_id",
    ],
    entity_rows = entity_rows
)

retrieved_df = feature_vector.to_df()
print(tabulate(retrieved_df, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

customer_id      days_since_last_order  favorite_category      avg_order_value    total_spent    total_orders    community_id
-------------  -----------------------  -------------------  -----------------  -------------  --------------  --------------
CUST_0001                            9  Electronics                     281.41        6190.93              22              99
CUST_0025                            0  Books                           248.23        7446.83              30              98
CUST_0050                            7  Sports                          289.29        7521.60              26              91


## 13. Use Features for Prediction

We use the retrieved features for a simple churn risk prediction. The logic is `days_since_last_order` drives the risk tier, demonstrating that switching the feature computation backend doesn't change the serving interface at all.

In [14]:
def predict_customer_churn(customer_id):
    """Use features for real-time churn prediction."""

    features = fs.get_online_features(
        features = [
            "customer_features:total_orders",
            "customer_features:total_spent",
            "customer_features:days_since_last_order",
            "customer_features:community_id",
        ],
        entity_rows = [{"customer_id": customer_id}]
    ).to_df()

    days_since = features["days_since_last_order"].iloc[0]
    community  = features["community_id"].iloc[0]

    if days_since > 60:
        risk, action = "HIGH",   "Send 20% discount"
    elif days_since > 30:
        risk, action = "MEDIUM", "Send personalized email"
    else:
        risk, action = "LOW",    "Continue engagement"

    return {
        "customer_id":  customer_id,
        "churn_risk":   risk,
        "action":       action,
        "community_id": int(community),
    }

predictions = [predict_customer_churn(c) for c in ["CUST_0001", "CUST_0025", "CUST_0050"]]
print(tabulate(predictions, headers = "keys", tablefmt = "simple"))

customer_id    churn_risk    action                 community_id
-------------  ------------  -------------------  --------------
CUST_0001      LOW           Continue engagement              99
CUST_0025      LOW           Continue engagement              98
CUST_0050      LOW           Continue engagement              91


## 14. Query Historical Features

We can also query historical features directly from Neo4j, restricting to orders placed before a given date.

In [15]:
def get_historical_features(customer_id, as_of_date):
    """Get features as they existed at a specific date."""

    with driver.session(database = NEO4J_DATABASE) as session:
        result = session.run("""
            MATCH (c:Customer {id: $customer_id})-[:PLACED]->(o:Order)
            WHERE o.timestamp <= datetime($as_of_date)
            RETURN
                c.id         AS customer_id,
                count(o)     AS total_orders,
                sum(o.value) AS total_spent,
                avg(o.value) AS avg_order_value
        """, customer_id = customer_id, as_of_date = as_of_date)
        df = pd.DataFrame([r.data() for r in result])

    if df.empty:
        print(f"No historical data found for {customer_id} as of {as_of_date}")
        return df

    return df

past_date = (datetime.now() - timedelta(days = 30)).strftime("%Y-%m-%dT%H:%M:%S")
hist_df   = get_historical_features("CUST_0001", past_date)

print(f"Features for CUST_0001 as of {past_date}:")
print()
print(tabulate(hist_df, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

Features for CUST_0001 as of 2026-08-01T08:22:11:

customer_id      total_orders    total_spent    avg_order_value
-------------  --------------  -------------  -----------------
CUST_0001                  18        5055.67             280.87


## 15. Teardown

In [16]:
driver.close()
print("Driver closed.")

Driver closed.
